In [ ]:
# 1 

# Tiandra Taylor


In [21]:
# pre-works for 2
!pip install pymssql

In [60]:
# 2

# connect to db
import pymssql
	
conn = pymssql.connect (
	host=os.getenv('DB_HOST'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
	database = 'overland')
	
cursor = conn.cursor()

# import pd
import pandas as pd

# sum case statements to select to filter out nulls when less than 18 then 1 else 0 or something and add order by
# query to df 
query = query = """
WITH guest_counts AS (
    SELECT HHID, COUNT(GuestID) as Members, 
           SUM(CASE WHEN SeasonPass = 'y' THEN 1 ELSE 0 END) as SPCount
    FROM Guest
    GROUP BY HHID
),
age_counts AS (
    SELECT HHID, SUM(CASE WHEN Age IS NULL THEN 0 WHEN Age < 18 THEN 1 ELSE 0 END) as U18Count
    FROM Guest
    GROUP BY HHID
),
visit_counts AS (
    SELECT g.HHID, COUNT(v.VisID) as VisitCt
    FROM Guest g
    JOIN Visit v ON g.GuestID = v.CustID
    GROUP BY g.HHID
)
SELECT h.HHID, h.Region, h.HHI as Income, g.Lname, gc.Members, gc.SPCount, ac.U18Count, vc.VisitCt
FROM Household h
JOIN guest_counts gc ON h.HHID = gc.HHID
JOIN Guest g ON h.HHID = g.HHID
LEFT JOIN age_counts ac ON h.HHID = ac.HHID
LEFT JOIN visit_counts vc ON h.HHID = vc.HHID
GROUP BY h.HHID, h.Region, h.HHI, g.Lname, gc.Members, gc.SPCount, ac.U18Count, vc.VisitCt
ORDER BY h.HHID
"""

df = pd.read_sql(query, conn, index_col='HHID')
print(df)

# I did have Claude help me fix my query. I couldn't figure out how to write 'y' in the query because it kept erroring (because it was already surrounded in single quotations).
# I also had it help me fix some of my syntax because I kept getting an error and I couldn't figure out why. I was missing some commas between my CTE's
# I also had it help check my age count code to make sure it was being calculated correctly because I had lots of null values so it explained that if I changed the values to 1's and 0's and summed them up it would work

C:\Users\Tiand\AppData\Local\Temp\ipykernel_28796\1979715992.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, index_col='HHID')


          Region  Income        Lname  Members  SPCount  U18Count  VisitCt
HHID                                                                      
1     Western PA  153000       Willis        2        0         1     12.0
2     Western PA  129000      Simpson        3        0         2      6.0
3          Other   68000  Sonic-Smith        1        0         0      6.0
4     Central PA   98000      Elliott        1        0         0      3.0
5     Central PA   52000        Owens        1        1         0      6.0
...          ...     ...          ...      ...      ...       ...      ...
7151  Central PA   36000     Mitchell        1        0         0      2.0
7152  Western PA   85000     Mitchell        1        0         0      6.0
7153     DC Area   72000        Green        1        0         0      5.0
7154  Central PA   48000    Schroeder        1        0         0      3.0
7155  Western PA  111000         Carr        1        0         1      3.0

[7155 rows x 7 columns]


In [61]:
# 3

# imports 
import numpy as np

# no nulls & df to console
df['SPCount'].replace(to_replace=np.nan, value = 0, inplace = True)
df['U18Count'].replace(to_replace=np.nan, value = 0, inplace = True)
df['VisitCt'].replace(to_replace=np.nan, value = 0, inplace = True)

print(df)

          Region  Income        Lname  Members  SPCount  U18Count  VisitCt
HHID                                                                      
1     Western PA  153000       Willis        2        0         1     12.0
2     Western PA  129000      Simpson        3        0         2      6.0
3          Other   68000  Sonic-Smith        1        0         0      6.0
4     Central PA   98000      Elliott        1        0         0      3.0
5     Central PA   52000        Owens        1        1         0      6.0
...          ...     ...          ...      ...      ...       ...      ...
7151  Central PA   36000     Mitchell        1        0         0      2.0
7152  Western PA   85000     Mitchell        1        0         0      6.0
7153     DC Area   72000        Green        1        0         0      5.0
7154  Central PA   48000    Schroeder        1        0         0      3.0
7155  Western PA  111000         Carr        1        0         1      3.0

[7155 rows x 7 columns]


C:\Users\Tiand\AppData\Local\Temp\ipykernel_28796\4004876372.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SPCount'].replace(to_replace=np.nan, value = 0, inplace = True)
C:\Users\Tiand\AppData\Local\Temp\ipykernel_28796\4004876372.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

In [62]:
# 4 

# visPP col for vis count per pp in household & df Lname, Members, VisitCt, and VisPP to console
df['VisPP'] = df['VisitCt'] / df['Members']


print(df[['Lname', 'Members', 'VisitCt', 'VisPP']])
# I had to ask Claude what exactly this was asking because I ran a query that counted Vis ID by guest ID 
# and Christian (person in HHID 1) had a count of 7 & Xavier (other person in HHID 1) had only 5. Our DF 
# is set up to have only one row per HHID so I was confused on what to do because we couldn't add that to 
#a df by one single row

            Lname  Members  VisitCt  VisPP
HHID                                      
1          Willis        2     12.0    6.0
2         Simpson        3      6.0    2.0
3     Sonic-Smith        1      6.0    6.0
4         Elliott        1      3.0    3.0
5           Owens        1      6.0    6.0
...           ...      ...      ...    ...
7151     Mitchell        1      2.0    2.0
7152     Mitchell        1      6.0    6.0
7153        Green        1      5.0    5.0
7154    Schroeder        1      3.0    3.0
7155         Carr        1      3.0    3.0

[7155 rows x 4 columns]


In [64]:
# 5

# imports
import numpy as np

# new df IsLocal if Central PA then 1 else 0 | df Lname, Region, and IsLocal to console
conditions = [(df['Region'] == 'Central PA'),
             (df['Region'] != 'Central PA')]

values = [1,0]

df['IsLocal'] = np.select(conditions,values)              
print(df[['Lname', 'Region', 'IsLocal']])

            Lname      Region  IsLocal
HHID                                  
1          Willis  Western PA        0
2         Simpson  Western PA        0
3     Sonic-Smith       Other        0
4         Elliott  Central PA        1
5           Owens  Central PA        1
...           ...         ...      ...
7151     Mitchell  Central PA        1
7152     Mitchell  Western PA        0
7153        Green     DC Area        0
7154    Schroeder  Central PA        1
7155         Carr  Western PA        0

[7155 rows x 3 columns]


In [65]:
# 6

# dfcopy = dflocal where IsLocal = 1

dfLocal = df[df['IsLocal'] == 1]

print(dfLocal)

          Region  Income      Lname  Members  SPCount  U18Count  VisitCt  \
HHID                                                                       
4     Central PA   98000    Elliott        1        0         0      3.0   
5     Central PA   52000      Owens        1        1         0      6.0   
6     Central PA  159000  Underwood        2        2         1     18.0   
7     Central PA  138000       Wise        1        0         1      5.0   
8     Central PA   78000      Tyler        6        0         4     22.0   
...          ...     ...        ...      ...      ...       ...      ...   
7143  Central PA   80000     Molina        1        0         0      9.0   
7145  Central PA  134000      Scott        5        0         4     33.0   
7149  Central PA  139000     Watson        2        0         0     71.0   
7151  Central PA   36000   Mitchell        1        0         0      2.0   
7154  Central PA   48000  Schroeder        1        0         0      3.0   

          V

In [68]:
# 7

#delete islocal and region col from dflocal df to console
dfLocal = dfLocal.drop(columns=['IsLocal', 'Region'])
print(dfLocal)                       

      Income      Lname  Members  SPCount  U18Count  VisitCt      VisPP
HHID                                                                   
4      98000    Elliott        1        0         0      3.0   3.000000
5      52000      Owens        1        1         0      6.0   6.000000
6     159000  Underwood        2        2         1     18.0   9.000000
7     138000       Wise        1        0         1      5.0   5.000000
8      78000      Tyler        6        0         4     22.0   3.666667
...      ...        ...      ...      ...       ...      ...        ...
7143   80000     Molina        1        0         0      9.0   9.000000
7145  134000      Scott        5        0         4     33.0   6.600000
7149  139000     Watson        2        0         0     71.0  35.500000
7151   36000   Mitchell        1        0         0      2.0   2.000000
7154   48000  Schroeder        1        0         0      3.0   3.000000

[2877 rows x 7 columns]


In [73]:
# 8

# no null records using .dropna would be fastest however that is not specified in the book
dfLocal = dfLocal[(dfLocal['Income'] != np.nan) & 
                  (dfLocal['Lname'] != np.nan) &
                  (dfLocal['Members'] != np.nan) &
                  (dfLocal['SPCount'] != np.nan) &
                  (dfLocal['U18Count'] != np.nan) &
                  (dfLocal['VisitCt'] != np.nan) &
                  (dfLocal['VisPP'] != np.nan)]
                    

# new col [IncomeCat] df local -> np.select ...df Lname, Income, IncomeCate, and VisPP to console

conditions = [(dfLocal['Income'] < 50000),
              ((dfLocal['Income'] >= 50000) & (dfLocal['Income'] < 100000)),
              ((dfLocal['Income'] >= 100000) & (dfLocal['Income'] < 150000)),
              (dfLocal['Income'] >= 150000)]

values = ['Under $50K', '$50-100K', '$100-150k', 'Over $150K']

dfLocal['IncomeCat'] = np.select(conditions,values)

print(dfLocal[['Lname', 'Income', 'IncomeCat', 'VisPP']])

          Lname  Income   IncomeCat      VisPP
HHID                                          
4       Elliott   98000    $50-100K   3.000000
5         Owens   52000    $50-100K   6.000000
6     Underwood  159000  Over $150K   9.000000
7          Wise  138000   $100-150k   5.000000
8         Tyler   78000    $50-100K   3.666667
...         ...     ...         ...        ...
7143     Molina   80000    $50-100K   9.000000
7145      Scott  134000   $100-150k   6.600000
7149     Watson  139000   $100-150k  35.500000
7151   Mitchell   36000  Under $50K   2.000000
7154  Schroeder   48000  Under $50K   3.000000

[2877 rows x 4 columns]


In [82]:
# 9

# rel betwee num of visPP & income? group by & agg for local only
print(dfLocal[['IncomeCat', 'VisPP']].groupby('IncomeCat').agg('mean'))

# print statement
print("\n","There is a weak, positive correlation between income and the number of visits per person in each houeshold. Correlation is not casuation, and there is a huge emphasis on 'weak'")

               VisPP
IncomeCat           
$100-150k   6.030133
$50-100K    4.806644
Over $150K  7.532885
Under $50K  3.891525

 There is a weak, positive correlation between income and the number of visits per person in each houeshold. Correlation is not casuation, and there is a huge emphasis on 'weak'


In [95]:
# 10
# this kept giving me an error saying it couldn't convert 'Elliot' to a float 
# I did some research and .corr is supposed to ignore columns labelled as "object" but obviously 
# that wasn't the case. I tried to specify that Lname and IncomeCat were category columns & ran it
# but it still gave me the same error so we turned to claude


# exclude visitct make correlation matrix cormat to console
dfCorr = dfLocal.drop(columns=['VisitCt'])

# I got this code from Claude but essentially what it is doing is specify which columns are numerical, 
# then running a correlation matrix for only the numerical columns and printing it out

numeric_cols = dfCorr.select_dtypes(include=[np.number]).columns
correlation_matrix = dfCorr[numeric_cols].corr()
print(correlation_matrix)

# strongest visPP & weakest visPP rel
print("\n","Income has the strongest relationship to VisPP (because it is closest to 1 or -1) & Members has the weakest relationship with VisPP (because it is the closest to 0).") 

            Income   Members   SPCount  U18Count     VisPP
Income    1.000000  0.350502  0.396300  0.291299  0.285967
Members   0.350502  1.000000  0.193409  0.902988 -0.118143
SPCount   0.396300  0.193409  1.000000  0.168820  0.233857
U18Count  0.291299  0.902988  0.168820  1.000000 -0.119280
VisPP     0.285967 -0.118143  0.233857 -0.119280  1.000000

 Income has the strongest relationship to VisPP (because it is closest to 1 or -1) & Members has the weakest relationship with VisPP (because it is the closest to 0).
